In [2]:
import os
from pathlib import Path
import json
import pandas as pd

# Always set project root as working dir (so relative paths work)
PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "data").exists() is False and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent  # you are likely inside /notebooks

os.chdir(PROJECT_ROOT)

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "raw_credit_applications.json"
RAW_PATH


PosixPath('/Users/riccardobertolini9/Desktop/project-teamX/data/raw/raw_credit_applications.json')

In [3]:
with open(RAW_PATH, "r") as f:
    data = json.load(f)

len(data), type(data), type(data[0])

(502, list, dict)

In [4]:
import pandas as pd

In [5]:
import json

path = "data/raw/raw_credit_applications.json"

with open(path, "r") as f:
    data = json.load(f)

len(data)

502

In [6]:
import pandas as pd

df = pd.json_normalize(data)
df.shape

(502, 21)

In [7]:
df.head(3)

,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,...,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,...,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,...,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,...,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN


In [8]:
df.columns.tolist()[:10]

['_id',
 'spending_behavior',
 'processing_timestamp',
 'applicant_info.full_name',
 'applicant_info.email',
 'applicant_info.ssn',
 'applicant_info.ip_address',
 'applicant_info.gender',
 'applicant_info.date_of_birth',
 'applicant_info.zip_code']

In [9]:
# 1) Partiamo da id + spending_behavior
sp = df[["_id", "spending_behavior"]].copy()

# 2) Trasformiamo la lista in righe (explode)
sp = sp.explode("spending_behavior", ignore_index=True)

# 3) Espandiamo i dict dentro spending_behavior in colonne (category, amount)
sp_details = pd.json_normalize(sp["spending_behavior"])
sp = pd.concat([sp.drop(columns=["spending_behavior"]), sp_details], axis=1)

sp.head(10)

,_id,category,amount
0,app_200,Shopping,480
1,app_200,Rent,790
2,app_200,Alcohol,247
3,app_037,Rent,608
4,app_037,Dining,96
5,app_037,Healthcare,243
6,app_215,Rent,109
7,app_024,Fitness,575
8,app_184,Entertainment,463
9,app_275,Entertainment,571


In [10]:
sp.shape, sp.isna().mean().sort_values(ascending=False).head(10)

((827, 3),
 _id         0.0
 category    0.0
 amount      0.0
 dtype: float64)

In [11]:
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_pct

notes                               99.601594
financials.annual_salary            99.003984
loan_purpose                        90.039841
processing_timestamp                87.649402
decision.rejection_reason           58.167331
decision.approved_amount            41.832669
decision.interest_rate              41.832669
financials.annual_income             0.996016
applicant_info.ip_address            0.996016
applicant_info.ssn                   0.996016
applicant_info.date_of_birth         0.199203
applicant_info.zip_code              0.199203
applicant_info.gender                0.199203
spending_behavior                    0.000000
financials.credit_history_months     0.000000
financials.debt_to_income            0.000000
financials.savings_balance           0.000000
decision.loan_approved               0.000000
applicant_info.email                 0.000000
applicant_info.full_name             0.000000
_id                                  0.000000
dtype: float64

In [12]:
# quando loan NON è approvato
df[df["decision.loan_approved"] == False][
    ["decision.approved_amount", "decision.interest_rate"]
].isna().mean()

decision.approved_amount    1.0
decision.interest_rate      1.0
dtype: float64

In [13]:
# quando loan è approvato
df[df["decision.loan_approved"] == True][
    ["decision.approved_amount", "decision.interest_rate"]
].isna().mean()

decision.approved_amount    0.0
decision.interest_rate      0.0
dtype: float64

In [14]:
(df["financials.credit_history_months"] < 0).sum()

np.int64(2)

In [15]:
((df["financials.debt_to_income"] < 0) | 
 (df["financials.debt_to_income"] > 1)).sum()

np.int64(1)

In [16]:
df["applicant_info.date_of_birth"] = pd.to_datetime(
    df["applicant_info.date_of_birth"], errors="coerce"
)

df["age"] = (pd.Timestamp("today") - df["applicant_info.date_of_birth"]).dt.days / 365

In [17]:
(df["age"] < 18).sum(), (df["age"] > 100).sum()

(np.int64(0), np.int64(0))

In [18]:
df[df["financials.credit_history_months"] < 0][
    ["_id", "financials.credit_history_months"]
]

,_id,financials.credit_history_months
137,app_043,-10
162,app_156,-3


In [19]:
df[(df["financials.debt_to_income"] < 0) |
   (df["financials.debt_to_income"] > 1)][
    ["_id", "financials.debt_to_income"]
]

,_id,financials.debt_to_income
316,app_402,1.85


In [20]:
df.loc[df["financials.credit_history_months"] < 0,
       "financials.credit_history_months"] = pd.NA

In [21]:
(df["financials.credit_history_months"] < 0).sum()

np.int64(0)

In [22]:
df_apps = df.drop(columns=["spending_behavior"]).copy()
df_apps.shape

(502, 21)

In [23]:
import pandas as pd

# df_apps deve esistere: se non l'hai creato, fallo ora
df_apps = df.drop(columns=["spending_behavior"]).copy()

bad_cols = []
for c in df_apps.columns:
    try:
        pd.DataFrame({c: df_apps[c]}).to_parquet("/tmp/test.parquet", index=False)
    except Exception as e:
        bad_cols.append((c, type(e).__name__, str(e)[:200]))

bad_cols

[('financials.annual_income',
  'ArrowInvalid',
  '("Could not convert \'55000\' with type str: tried to convert to int64", \'Conversion failed for column financials.annual_income with type object\')')]

In [24]:
# forziamo annual_income a numeric: stringhe -> numeri, valori non convertibili -> NaN
df_apps["financials.annual_income"] = pd.to_numeric(
    df_apps["financials.annual_income"],
    errors="coerce"
)

df_apps["financials.annual_income"].dtype

dtype('float64')

In [25]:
df_apps["financials.annual_income"].isna().sum()

np.int64(5)

In [26]:
df_apps.to_parquet("data/processed/credit_applications_clean.parquet", index=False)
sp.to_parquet("data/processed/spending_behavior.parquet", index=False)

In [27]:
cols_to_remove = [
    "applicant_info.ssn",
    "applicant_info.ip_address"
]

df_clean = df_apps.drop(columns=cols_to_remove).copy()

df_clean.shape

(502, 19)

In [28]:
missing_ratio = df_clean.isna().mean()

cols_high_missing = missing_ratio[missing_ratio > 0.9].index.tolist()

cols_high_missing

['loan_purpose', 'financials.annual_salary', 'notes']

In [29]:
df_clean = df_clean.drop(columns=cols_high_missing)
df_clean.shape

(502, 16)

In [30]:
df_clean.to_parquet(
    "data/clean/credit_applications_final.parquet",
    index=False
)

In [31]:
def clean_credit_applications(df_raw):
    import pandas as pd
    
    df = pd.json_normalize(df_raw)
    
    # --- Remove nested column ---
    if "spending_behavior" in df.columns:
        df = df.drop(columns=["spending_behavior"])
    
    # --- Fix invalid values ---
    df.loc[df["financials.credit_history_months"] < 0,
           "financials.credit_history_months"] = pd.NA
    
    df.loc[
        (df["financials.debt_to_income"] < 0) |
        (df["financials.debt_to_income"] > 1),
        "financials.debt_to_income"
    ] = pd.NA
    
    # --- Fix type mismatch ---
    df["financials.annual_income"] = pd.to_numeric(
        df["financials.annual_income"],
        errors="coerce"
    )
    
    # --- Remove sensitive columns ---
    sensitive_cols = [
        "applicant_info.ssn",
        "applicant_info.ip_address"
    ]
    df = df.drop(columns=sensitive_cols, errors="ignore")
    
    # --- Remove high missing columns ---
    missing_ratio = df.isna().mean()
    high_missing = missing_ratio[missing_ratio > 0.9].index
    df = df.drop(columns=high_missing)
    
    return df

In [32]:
def clean_data(df):

    # --- Fix invalid debt_to_income ---
    df.loc[
        (df["financials.debt_to_income"] < 0) |
        (df["financials.debt_to_income"] > 1),
        "financials.debt_to_income"
    ] = pd.NA

    # --- Fix invalid credit history ---
    df.loc[
        df["financials.credit_history_months"] < 0,
        "financials.credit_history_months"
    ] = pd.NA

    # --- Fix income type mismatch ---
    df["financials.annual_income"] = pd.to_numeric(
        df["financials.annual_income"],
        errors="coerce"
    )

    # --- Remove sensitive columns ---
    sensitive_cols = [
        "applicant_info.ssn",
        "applicant_info.ip_address"
    ]
    df = df.drop(columns=sensitive_cols, errors="ignore")

    # --- Remove high missing columns ---
    missing_ratio = df.isna().mean()
    high_missing = missing_ratio[missing_ratio > 0.9].index
    df = df.drop(columns=high_missing)

    return df

In [33]:
df_clean = clean_data(df.copy())
df.shape, df_clean.shape

((502, 22), (502, 17))

In [34]:
# Duplicate check solo su _id (corretto)
dup_id = df_clean["_id"].duplicated().sum()

dup_id

np.int64(2)

In [35]:
df_clean["_id"].duplicated().sum()

np.int64(2)

In [36]:
df_clean[df_clean["_id"].duplicated(keep=False)].sort_values("_id")

,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,decision.interest_rate,decision.approved_amount,age
383,app_001,"[{'category': 'Fitness', 'amount': 576}]",NaN,Stephanie Nguyen,stephanie.nguyen47@mail.com,Female,1986-05-27,90230,102000.0,37.0,0.42,0,False,high_dti_ratio,NaN,NaN,39.778082
455,app_001,"[{'category': 'Fitness', 'amount': 576}]",NaN,Stephanie Nguyen,stephanie.nguyen47@mail.com,NaN,NaT,NaN,102000.0,37.0,0.42,0,False,high_dti_ratio,NaN,NaN,NaN
8,app_042,"[{'category': 'Insurance', 'amount': 153}, {'c...",NaN,Joseph Lopez,joseph.lopez1@gmail.com,Male,1990-05-04,10044,69000.0,43.0,0.41,15974,False,algorithm_risk_score,NaN,NaN,35.838356
354,app_042,"[{'category': 'Insurance', 'amount': 153}, {'c...",NaN,Joseph Lopez,joseph.lopez1@gmail.com,Male,1990-05-04,10044,69000.0,43.0,0.41,15974,False,algorithm_risk_score,NaN,NaN,35.838356


In [37]:
df_clean = df_clean.drop_duplicates(subset=["_id"], keep="first").copy()

df_clean["_id"].duplicated().sum()

np.int64(0)

### Data Quality Summary

The following issues were identified and addressed:

- 2 duplicate application IDs → resolved by keeping first occurrence.
- Invalid credit history values (<0) → replaced with NA.
- Debt-to-income ratios outside valid range [0,1] → replaced with NA.
- Annual income stored as string in some records → converted to numeric.
- Columns with >90% missing values removed.
- Sensitive personal identifiers (SSN, IP address) removed.

Final dataset:
- 502 records
- 17 columns
- No duplicate IDs
- Cleaned and exported to parquet format

In [38]:
df_clean.to_parquet(
    "data/clean/credit_applications_final.parquet",
    index=False
)

In [39]:
import os

os.path.exists("data/clean/credit_applications_final.parquet"), os.path.getsize("data/clean/credit_applications_final.parquet")

(True, 43786)

In [42]:
print("Final dataset shape:", df_clean.shape)

print("\nMissing values per column:")
print(df_clean.isnull().sum())

print("\nDuplicate application IDs:",
      df_clean["_id"].duplicated().sum())

print("\nBasic statistics:")
print(df_clean.describe())

Final dataset shape: (500, 17)

Missing values per column:
_id                                   0
spending_behavior                     0
processing_timestamp                438
applicant_info.full_name              0
applicant_info.email                  0
applicant_info.gender                 0
applicant_info.date_of_birth        161
applicant_info.zip_code               0
financials.annual_income              5
financials.credit_history_months      2
financials.debt_to_income             1
financials.savings_balance            0
decision.loan_approved                0
decision.rejection_reason           292
decision.interest_rate              208
decision.approved_amount            208
age                                 161
dtype: int64

Duplicate application IDs: 0

Basic statistics:
        applicant_info.date_of_birth  financials.annual_income  \
count                            339                495.000000   
mean   1984-08-23 09:29:12.212389376              82693.803614   
m

In [43]:
df_clean = df_clean.drop(columns=["processing_timestamp"])

In [44]:
df_clean.to_parquet(
    "data/clean/credit_applications_clean_v2.parquet",
    index=False
)

In [45]:
import os
os.path.exists("data/clean/credit_applications_clean_v2.parquet")

True